# AGML-DenseCBAM — Leakage-Safe Training

This notebook is a thin interface to the active Python pipeline. Keeping model and evaluation logic in one implementation prevents the notebook and scripts from drifting apart.

Final workflow: audit data → generate leakage-safe folds → smoke test → run C1–C4 → aggregate Chapter IV results.

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'train_one_case_5fold.py').exists():
    raise RuntimeError('Launch Jupyter from the AGML-DenseCBAM project root.')
DATA_ROOT = PROJECT_ROOT / 'data'
FOLDS_ROOT = PROJECT_ROOT / 'data_5_fold'
RESULTS_ROOT = PROJECT_ROOT / 'chapter4_results' / 'final_3002_seed42'

# Safety switches: explicitly enable only the stages you intend to run.
REGENERATE_FOLDS = False  # set True only after exclusion approval
RUN_SMOKE_TESTS = False   # set True to run a few cases for debugging
RUN_FINAL_CASES = True    # set True after smoke tests pass
RUN_ANALYSIS = True       # set True with final cases or after all cases finish
SKIP_EXISTING = True      # safely resumes only matching code/config/folds

print('Python:', sys.version)
print('Executable:', sys.executable)
print('Project:', PROJECT_ROOT)
print('Final results:', RESULTS_ROOT)

Python: 3.13.13 | packaged by conda-forge | (main, Apr  8 2026, 02:00:33) [GCC 14.3.0]
Executable: /home/solyvie/environments/miniforge3/envs/thesis-env/bin/python
Project: /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM
Final results: /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42


## 1. Audit and generate folds

The default `error` policy writes `duplicate_audit.csv` and stops when identical pixels have conflicting labels. Review those rows with the dataset owner. Change to `exclude` only when exclusion has been formally approved and will be documented in the thesis. Add `--groups_csv ...` when patient/study IDs are available.

In [2]:
CONFLICT_POLICY = 'error'  # change to 'exclude' only after review/approval
if REGENERATE_FOLDS:
    fold_command = [
        sys.executable, 'make_5fold_dataset.py',
        '--input_root', str(DATA_ROOT),
        '--output_root', str(FOLDS_ROOT),
        '--n_splits', '5', '--val_size', '0.15', '--seed', '42',
        '--conflict_policy', CONFLICT_POLICY,
    ]
    subprocess.run(fold_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Fold regeneration skipped; using:', FOLDS_ROOT)

Fold regeneration skipped; using: /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/data_5_fold


## 2. Smoke test

This uses one fold and two epochs in a separate output directory. It is not a reportable result.

In [3]:
SMOKE_CASES = [('benchmark', 'clean'), ('proposed', 'artifact_mix')]
if RUN_SMOKE_TESTS:
    for model_type, scenario in SMOKE_CASES:
        smoke_command = [
            sys.executable, 'run_case_5fold_isolated.py',
            '--folds_root', str(FOLDS_ROOT),
            '--model_type', model_type, '--scenario', scenario,
            '--epochs', '1', '--batch_size', '8', '--fold_limit', '1',
            '--output_dir', str(PROJECT_ROOT / 'chapter4_results' / 'smoke' / f'{model_type}_{scenario}'),
        ]
        if SKIP_EXISTING:
            smoke_command.append('--skip_existing')
        print('\nSMOKE TEST:', model_type, scenario)
        subprocess.run(smoke_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Smoke tests skipped.')

Smoke tests skipped.


## 3. Final C1–C4 runs

Run all cases on the same folds, seed, hyperparameters, and hardware. A fresh process is used for each fold. This can take many hours.

In [4]:
EPOCHS = 50
CASES = [
    ('benchmark', 'clean'),
    ('benchmark', 'artifact_mix'),
    ('proposed', 'clean'),
    ('proposed', 'artifact_mix'),
]

if RUN_FINAL_CASES:
    for model_type, scenario in CASES:
        output_dir = RESULTS_ROOT / f'{model_type}_{scenario}'
        command = [
            sys.executable, 'run_case_5fold_isolated.py',
            '--folds_root', str(FOLDS_ROOT),
            '--model_type', model_type, '--scenario', scenario,
            '--epochs', str(EPOCHS), '--batch_size', '8',
            '--learning_rate', '1e-4', '--l2_strength', '1e-2',
            '--seed', '42', '--output_dir', str(output_dir),
        ]
        if SKIP_EXISTING:
            command.append('--skip_existing')
        print('\nFINAL CASE:', model_type, scenario)
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('Final C1-C4 runs are disabled. Set RUN_FINAL_CASES = True after smoke tests pass.')


FINAL CASE: benchmark clean

=== Isolated fold process ===
/home/solyvie/environments/miniforge3/envs/thesis-env/bin/python /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/train_one_case_5fold.py --folds_root /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/data_5_fold --output_dir /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_clean --model_type benchmark --scenario clean --image_size 224 --batch_size 8 --epochs 50 --learning_rate 0.0001 --l2_strength 0.01 --seed 42 --tmd_loss_weight 1.0 --artifact_loss_weight 0.3 --single_fold fold_1


I0000 00:00:1786771135.246184   12409 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786771137.242473   12409 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow precision policy: mixed_float16
TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Config: RunConfig(folds_root=PosixPath('/home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/data_5_fold'), output_dir=PosixPath('/home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_clean'), model_type='benchmark', scenario='clean', image_size=(224, 224), batch_size=8, epochs=50, learning_rate=0.0001, l2_strength=0.01, random_state=42, freeze_backbone=False, tmd_loss_weight=1.0, artifact_loss_weight=0.3, fold_limit=None, single_fold='fold_1', verify_integrity=True, mixed_precision=True, class_weighting=True)

=== Running benchmark | clean | fold_1 ===
split       class_name 
test        normal          272
            subluxation     329
train       normal          931
            subluxation    1127
validation  normal          156
            subluxation     187
dtype: int

E0000 00:00:1786771227.894421   12409 util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


258/258 ━━━━━━━━━━━━━━━━━━━━ 0s 592ms/step - accuracy: 0.5403 - loss: 9.0385
Epoch 1: val_loss improved from None to 4.60211, saving model to /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_clean/fold_1_benchmark_clean_best.keras

Epoch 1: finished saving model to /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_clean/fold_1_benchmark_clean_best.keras
258/258 ━━━━━━━━━━━━━━━━━━━━ 254s 653ms/step - accuracy: 0.5957 - loss: 7.3629 - val_accuracy: 0.7813 - val_loss: 4.6021 - learning_rate: 1.0000e-04
Epoch 2/50
258/258 ━━━━━━━━━━━━━━━━━━━━ 0s 583ms/step - accuracy: 0.8049 - loss: 3.8552
Epoch 2: val_loss improved from 4.60211 to 2.48951, saving model to /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_clean/fold_1_benchmark_clean_best.keras

Epoch 2: finished saving model to /home/solyvie/workspace/the

...

TensorFlow precision policy: mixed_float16
TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Config: RunConfig(folds_root=PosixPath('/home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/data_5_fold'), output_dir=PosixPath('/home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_artifact_mix'), model_type='benchmark', scenario='artifact_mix', image_size=(224, 224), batch_size=8, epochs=50, learning_rate=0.0001, l2_strength=0.01, random_state=42, freeze_backbone=False, tmd_loss_weight=1.0, artifact_loss_weight=0.3, fold_limit=None, single_fold='fold_4', verify_integrity=True, mixed_precision=True, class_weighting=True)

=== Running benchmark | artifact_mix | fold_4 ===
split       class_name 
test        normal          272
            subluxation     329
train       normal          932
            subluxation    1126
validation  normal          155
            subluxati

E0000 00:00:1786804137.792373  126341 util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


258/258 ━━━━━━━━━━━━━━━━━━━━ 0s 656ms/step - accuracy: 0.4935 - loss: 9.1844
Epoch 1: val_loss improved from None to 5.10209, saving model to /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_artifact_mix/fold_4_benchmark_artifact_mix_best.keras

Epoch 1: finished saving model to /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_artifact_mix/fold_4_benchmark_artifact_mix_best.keras
258/258 ━━━━━━━━━━━━━━━━━━━━ 332s 784ms/step - accuracy: 0.5097 - loss: 7.6546 - val_accuracy: 0.6152 - val_loss: 5.1021 - learning_rate: 1.0000e-04
Epoch 2/50
258/258 ━━━━━━━━━━━━━━━━━━━━ 0s 642ms/step - accuracy: 0.6330 - loss: 4.4281
Epoch 2: val_loss improved from 5.10209 to 2.84533, saving model to /home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_artifact_mix/fold_4_benchmark_artifact_mix_best.keras

Epoch 2: finished s

F0000 00:00:1786807060.838804  126476 reduction_gpu_kernels.cu.h:660] Check failed: (::tsl::TfCheckOkDeprecationMarker(), GpuLaunchKernel(BlockReduceKernel<IN_T, OUT_T, num_threads, Op>, num_blocks, num_threads, 0, cu_stream, in, out, in_size, op, init)) is OK (INTERNAL: unknown error) 
*** Check failure stack trace: ***
    @     0x70336391d314  absl::lts_20250814::log_internal::LogMessage::SendToLog()
    @     0x70336391d296  absl::lts_20250814::log_internal::LogMessage::Flush()
    @     0x70334a826c35  tensorflow::functor::LaunchScalarReduction<>()
    @     0x70334a82577d  tensorflow::functor::ReduceImpl<>()
    @     0x70334a825673  tensorflow::functor::ReduceFunctor<>::Reduce<>()
    @     0x70334a5e734d  tensorflow::ReductionOp<>::Compute()
    @     0x7033639876bc  tensorflow::BaseGPUDevice::Compute()
    @     0x703362947a29  tensorflow::(anonymous namespace)::ExecutorState<>::Process()
    @     0x7033635a8594  Eigen::ThreadPoolTempl<>::WorkerLoop()
    @     0x7033635a8411

CalledProcessError: Command '['/home/solyvie/environments/miniforge3/envs/thesis-env/bin/python', 'run_case_5fold_isolated.py', '--folds_root', '/home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/data_5_fold', '--model_type', 'benchmark', '--scenario', 'artifact_mix', '--epochs', '50', '--batch_size', '8', '--learning_rate', '1e-4', '--l2_strength', '1e-2', '--seed', '42', '--output_dir', '/home/solyvie/workspace/thesis-projects/working-run/AGML-DenseCBAM/chapter4_results/final_3002_seed42/benchmark_artifact_mix', '--skip_existing']' returned non-zero exit status 1.

## 4. Chapter IV aggregate analysis

The analyzer refuses incomplete case sets. Outputs include mean ± SD, 95% CIs, paired tests, robustness degradation comparisons, and graphs.

In [ ]:
if RUN_ANALYSIS:
    subprocess.run([
        sys.executable, 'analyze_chapter4.py',
        '--results_root', str(RESULTS_ROOT),
        '--expected_folds', '5',
    ], cwd=PROJECT_ROOT, check=True)
else:
    print('Analysis skipped. Enable RUN_ANALYSIS after all four final cases complete.')

## Interpretation safeguards

- Do not report `oldstyle_results` or smoke-test metrics as final results.
- State whether patient/study grouping was available. Hash deduplication alone does not establish patient independence.
- Describe artifacts as synthetic corruptions, not verified real clinical artifact classes.
- Five-fold inferential tests have low power; report fold values, confidence intervals, and effect sizes with p-values.
- Grad-CAM remains qualitative unless expert ROI annotations are available.